#CS-215 Final Project - Self Data Analysis

Hello!

This Google Colab notebook presents a self-data analysis project exploring my personal use of ChatGPT. Using real conversation data, it examines patterns in messaging behavior, including prompt length, usage frequency, active hours, and overall communication style.

The goal is to better understand how I interact with ChatGPT over time through data-driven insights and visualization. You're welcome to copy this notebook and apply it to your own data to explore your own usage patterns.

##Bringing Data

In [ ]:
#bring data from google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#glob is a tool that can find files based on a pattern
import glob
import pandas as pd

path = "/content/drive/MyDrive/CS215-Spring-Jenny Lim/Final Project/Final_Data/conversations-*.json"

files = glob.glob(path)

#read and combine all files
df = pd.concat((pd.read_json(f) for f in files), ignore_index=True)

In [ ]:
#Try to look at your data before pre-processing!
#df['mapping'].iloc[2]

##Data Pre-Processing

###1. Conversation Data Flattening and Cleaning Pipeline

This code is extracting and cleaning raw conversation data into a structured dataset.

It goes through each conversation in df, navigates the nested message structure (mapping), and pulls out only valid messages from users and assistants. For each message, it extracts the role, text content, and timestamp, while skipping system messages, empty text, and missing timestamps.

Finally, it stores everything in a clean list of dictionaries (rows), where each entry represents a single usable message ready for analysis.

In [ ]:
rows = []

for _, row in df.iterrows():
    convo_id = row["conversation_id"]
    mapping = row["mapping"]

    for node_id, node in mapping.items():
        message = node.get("message")

        if message is None:
            continue

        # extract relevant fields such as user/assistant, content, and time
        role = message.get("author", {}).get("role")
        content = message.get("content", {}).get("parts", [])
        time = message.get("create_time")

        # join all text parts into one string
        if isinstance(content, list):
            text = " ".join([str(p) for p in content if isinstance(p, str)])
        else:
            text = ""

        # remove system/empty messages/messages without timestamps
        if role not in ["user", "assistant"]:
            continue

        if not text.strip():
            continue

        if time is None:
            continue

        rows.append({
            "conversation_id": convo_id,
            "role": role,
            "text": text,
            "timestamp": time
        })

###2. Message Feature Engineering Pipeline (Time & Text Attributes)

This code converts the cleaned message list into a structured DataFrame and enriches it with additional features for analysis.

It first creates a DataFrame from rows, then converts the timestamp into a readable local time (Pacific Time). After that, it extracts useful time-based features such as date, hour, day of the week, and month, and also computes message length. Finally, it sorts the data by conversation and time to preserve the natural order of each chat before displaying the first few rows.

In [ ]:
messages_df = pd.DataFrame(rows)

messages_df["timestamp"] = (
    pd.to_datetime(messages_df["timestamp"], unit="s", utc=True)
      .dt.tz_convert("America/Los_Angeles")
      .dt.tz_localize(None)
)

messages_df["date"] = messages_df["timestamp"].dt.date
messages_df["hour"] = messages_df["timestamp"].dt.hour
messages_df["day_of_week"] = messages_df["timestamp"].dt.day_name()
messages_df["month"] = messages_df["timestamp"].dt.to_period("M")

messages_df["length"] = messages_df["text"].str.len()

messages_df = messages_df.sort_values(
    ["conversation_id", "timestamp"]
)

###3. User Message Filtering and Export for Analysis

This code filters the dataset to keep only user messages, then extracts time-based features like hour and day of the week from the timestamp. After preparing the cleaned user-only dataset, it saves the result as a CSV file and downloads it from Google Colab for further analysis or external use.

In [ ]:
user_df = messages_df[messages_df["role"] == "user"].copy()

In [ ]:
user_df["hour"] = user_df["timestamp"].dt.hour
user_df["day_of_week"] = user_df["timestamp"].dt.day_name()

In [ ]:
user_df.head()

,conversation_id,role,text,timestamp,date,hour,day_of_week,month,length
836,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,Week 7: Frictional and Normal Forces Problem (...,2025-03-05 14:32:43.802000046,2025-03-05,14,Wednesday,2025-03,1758
841,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,write up,2025-03-05 14:33:06.992000103,2025-03-05,14,Wednesday,2025-03,8
850,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,\tTrial 1\tTrial 2\tTrial 3\tTrial 4\tAverage\...,2025-03-05 14:59:52.551000118,2025-03-05,14,Wednesday,2025-03,449
838,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,angle was 0,2025-03-05 15:00:10.618000031,2025-03-05,15,Wednesday,2025-03,11
831,67c8d10c-09c8-8012-bbcb-2b2628150eca,user,\tTrial 1\tTrial 2\tTrial 3\tTrial 4\tAverage\...,2025-03-05 15:00:30.641000032,2025-03-05,15,Wednesday,2025-03,461


In [ ]:
#NOT NECESSARY IF YOU'RE ANALYZING YOUR OWN DATA
user_df.to_csv("math_engineerining_user_df.csv", index=False)

In [ ]:
#NOT NECESSARY IF YOU'RE ANALYZING YOUR OWN DATA

from google.colab import files
files.download("math_engineerining_user_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Visualization

###1. Usage by Hour

This code analyzes WHEN users are most active during the day.

It counts how many user prompts occur in each hour, then uses a bar chart to show the distribution of usage across the day. Each bar represents the number of prompts sent during a specific hour.

In [ ]:
import plotly.express as px

usage_by_hour = user_df["hour"].value_counts().sort_index()

fig1 = px.bar(
    x=usage_by_hour.index,
    y=usage_by_hour.values,
    labels={"x": "Hour", "y": "Number of Prompts"},
    title="Usage by Hour"
)

#fig1.write_html("hour.html")
fig1.show()

Key insight: The data shows a clear daily usage pattern with strong variation across different hours of the day. Activity is lowest in the early morning hours, particularly around 5~6 AM and 9 AM, indicating limited usage during typical sleeping or early routine times. Usage begins to increase gradually through the morning and rises more noticeably in the afternoon. The highest levels of activity occur in the late afternoon and evening, especially between 3 PM and 11 PM, with the peak at 11 PM (313 messages) followed closely by 10 PM and 4 PM. Overall, the pattern suggests that I am most active later in the day, with a strong concentration of usage during evening and late-night hours, likely reflecting study or assignment-related work done outside of regular daytime schedules.

###2. Usage by Date

This code is showing how user activity changes over calendar dates (day-by-day usage over time).

It first counts how many prompts were made on each date, then creates a bar chart where each bar represents the total number of user messages on a specific day.

In [ ]:
usage_by_date= user_df["date"].value_counts().sort_index()

fig2 = px.bar(
    x=usage_by_date.index,
    y=usage_by_date.values,
    labels={"x": "Date", "y": "Prompts"},
    title="Daily Usage Over Time",
    width=1200,
    height=500
)

fig2.update_traces(marker_line_width=0)
fig2.update_layout(bargap=0.2)

fig2.show()

Key insight: This data closely reflects my academic schedule and breaks throughout the year. My ChatGPT usage drops significantly during major breaks, including mid-May to July, October break, end of December through January (winter break), and spring break around March, where there is very little to no interaction. In contrast, during active school periods, usage is generally higher, showing a strong connection to coursework and academic needs. However, the data also shows inconsistency on a daily level—when I do engage with ChatGPT, some days involve heavy usage with many prompts, while other days show little to no interaction at all. This suggests that my usage is not only seasonal, but also highly variable depending on workload, deadlines, and specific academic demands on any given day.

###3. Correlation between user prompt and ChatGPT response

This code is analyzing the relationship between how long a user prompt is and how long the corresponding ChatGPT response is.

It first groups the dataset by conversation and looks at each message in order. Whenever it finds a user message followed immediately by an assistant response, it records the length of both texts. These pairs are stored in a new dataset called pairs_df

In [ ]:
correlation = []

for convo_id, group in messages_df.groupby("conversation_id"):
    group = group.reset_index(drop=True)

    for i in range(len(group) - 1):
        if group.loc[i, "role"] == "user" and group.loc[i+1, "role"] == "assistant":
            correlation.append({
                "prompt_length": group.loc[i, "length"],
                "response_length": group.loc[i+1, "length"]
            })

pairs_df = pd.DataFrame(correlation)
pairs_df.corr()

,prompt_length,response_length
prompt_length,1.000000,0.235979
response_length,0.235979,1.000000


In [ ]:
fig8 = px.scatter(
    pairs_df,
    x="prompt_length",
    y="response_length",
    title="Prompt vs Response Length",
    labels={
        "prompt_length": "Prompt Length",
        "response_length": "Response Length"
    }
)

#fig8.write_html("prompt_vs_response.html") --> use this if you want to download the data
fig8.show()

Key insights: The correlation between prompt length and response length is 0.236, which indicates a weak positive relationship. This suggests that there is a slight tendency for longer user prompts to receive longer responses, but the relationship is not strong enough to imply any consistent pattern. In other words, prompt length does not significantly determine how long the assistant's response will be. Instead, response length is likely influenced more by other factors such as the complexity of the question, the type of request being made, and the contextual needs of the conversation. Overall, this result shows that ChatGPT's response length is not heavily dependent on how much the user writes, but rather on the substance and nature of the prompt.

###4. Text Clustering to find hidden patterns

This code is performing text clustering to discover hidden patterns in user prompts, then visualizing how different types of usage are distributed.

First, it converts all user messages into numerical form using TF-IDF, which represents each text based on important words while reducing noise. Then it applies K-Means clustering (k=5) to automatically group similar messages together based on their word patterns.

After clustering, it prints sample messages from each cluster so I can interpret what each group represents. These clusters are then manually labeled with meaningful names like academic work, writing tasks, conceptual questions, and personal/narrative content.

Finally, it groups the data by these cluster labels and creates a treemap visualization, where each block shows how large each type of usage is. The result gives a high-level overview of the main categories of ChatGPT usage and how frequently each type appears in the dataset.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(user_df["text"])

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
user_df["cluster"] = kmeans.fit_predict(X)

In [ ]:
for c in sorted(user_df["cluster"].unique()):
    print("\n------------")
    print(f"CLUSTER {c}")
    print("------------")
    print(user_df[user_df["cluster"] == c]["text"].head().values)


------------
CLUSTER 0
------------
['write up'
 '\tTrial 1\tTrial 2\tTrial 3\tTrial 4\tAverage\tStd Dev.\nStatic frictional force (N)\t3.37\t3.13\t3.24\t3.27\t3.2525\t0.09878427675\nKinetic frictional force (N)\t2.79\t2.78\t2.8\t2.7\t2.7675\t0.04573474245\n\t\t\t\t\t\t\n\t\t\t\t\t\t\nNormal Force = m*g\t\t\t\t\t\t\nm (kg)\tg (m/s^2)\t\t\t\t\t\n1.6343\t9.8\t16.01614\t\t\t\t\n\t\t\t\t\t\t\nStatic friction coefficient\t\tUncertainty\t\t\t\t\n0.2030763967\t\t0.006167795533\t\t\t\t\n\t\t\t\t\t\t\nKinetic friction coefficient\t\tUncertainty\t\t\t\t\n0.1727944436\t\t0.002855540876\t\t\t\t\n\t\t\t\t\t\t'
 'angle was 0'
 '\tTrial 1\tTrial 2\tTrial 3\tTrial 4\tAverage\tStd Dev.\nStatic frictional force (N)\t3.37\t3.13\t3.24\t3.27\t3.2525\t0.09878427675\nKinetic frictional force (N)\t2.79\t2.78\t2.8\t2.7\t2.7675\t0.04573474245\n\t\t\t\t\t\t\n\t\t\t\t\t\t\nNormal Force = m*g\t\t\t\t\t\t\nm (kg)\tg (m/s^2)\t\t\t\t\t\n1.6343\t9.8\t16.01614\t\t\t\t\n\t\t\t\t\t\t\nStatic friction coefficient\t\tUnce

In [ ]:
cluster_map = {
    0: "Experimental / Physics Lab Data & Calculations",
    1: "Academic Assignments & Technical Course Material",
    2: "Short Questions / Clarifications / Concept Checks",
    3: "Personal Updates & Narrative Writing",
    4: "Writing Requests & Content Rewriting Tasks",
}

user_df["cluster_label"] = user_df["cluster"].map(cluster_map)

In [ ]:
treemap_df = user_df.groupby("cluster_label").size().reset_index(name="count")

In [ ]:
fig10 = px.treemap(
    treemap_df,
    path=["cluster_label"],
    values="count",
    color="count",
    color_continuous_scale="Blues",
    title="ChatGPT Usage Clusters (Treemap)"
)

fig10.show()

Key insights: The clustering results show that the majority of ChatGPT usage is concentrated in academically focused tasks, with "Experimental / Physics Lab Data & Calculations" (1,622 messages) and "Academic Assignments & Technical Course Material" (935 messages) being the two largest groups. This indicates that ChatGPT is primarily used as a support tool for STEM-related coursework and technical problem-solving. The third largest category, "Personal Updates & Narrative Writing" (880 messages), shows that ChatGPT is also used for more reflective or expressive tasks, but to a lesser extent than academic work. Smaller portions of usage fall under "Short Questions / Clarifications / Concept Checks" (397 messages), suggesting occasional use for quick learning or clarification, and "Writing Requests & Content Rewriting Tasks" (144 messages), which is the least frequent category, indicating that formal writing assistance is not a major part of usage. Overall, the distribution highlights a strong focus on technical and academic problem-solving, with relatively limited use for writing-heavy or casual assistance tasks.

###5. Heatmap

This code is creating a usage heatmap that shows when ChatGPT is used across both time of day and day of the week.

I first built a pivot table that counts how many messages were sent for each combination of day of the week (rows) and hour of the day (columns). The days are then reordered from Monday to Sunday, and missing values are filled with zero so the table is complete. Finally, the data is visualized as a heatmap where darker or more intense colors represent higher usage.

In [ ]:
heatmap_data = user_df.pivot_table(
    index="day_of_week",
    columns="hour",
    values="text",
    aggfunc="count"
)

days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap_data = heatmap_data.reindex(days_order)
heatmap_data = heatmap_data.fillna(0)

In [ ]:
fig7 = px.imshow(
    heatmap_data,
    labels=dict(x="Hour of Day", y="Day of Week", color="Number of Prompts"),
    title="ChatGPT Usage Heatmap"
)
fig7.update_xaxes(tickmode='linear')
#fig7.write_html("heatmap.html") --> use this if you want to download the data
fig7.show()

Key insights: The heatmap reveals a clear structure in ChatGPT usage across both days of the week and hours of the day, showing that activity is highly uneven and strongly concentrated during specific time periods. Weekdays (Monday through Friday) account for the majority of usage, with consistently higher counts across most hours, especially in the late afternoon and evening. In particular, Thursday and Friday show very strong late-night activity, with notable spikes such as 90 messages at 11 PM on Thursday and elevated usage around 3-4 PM and 10-11 PM across multiple weekdays.

In contrast, weekend usage is much lower overall, especially on Saturday, which shows minimal engagement throughout most hours. Sunday is an exception, showing a sharp increase in late-night activity, particularly at 10-11 PM, suggesting a return to academic or task-related work before the start of the week. Overall, the pattern suggests that ChatGPT usage is heavily tied to the academic schedule, with intensive weekday engagement and reduced weekend activity, while late evenings consistently represent the peak usage window across nearly all days.

##Project Main Notebook

After completing this step, I repeated the process five additional times, resulting in six distinct datasets for comparison. From this point onward…

https://colab.research.google.com/drive/1p56ZvCHkly8PDzFBaJQFLgGiqwWrEhiA?usp=chrome_ntp#scrollTo=zbi0-Nz84-iS

